Understanding how Spark executes lazily and runs only on actions

By now you’ve written a fair amount of Spark code. You’ve selected columns, added new ones, filtered rows, maybe even cleaned up some missing values. But here’s a curious detail: Spark hasn’t actually done anything yet.

That sounds strange at first. You typed a command, so shouldn’t Spark have run it? The answer is no — and understanding why is the key to becoming a confident Spark data engineer.

### Why Spark Waits
Spark follows what we call a lazy execution model. When you type something like:

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("day7-demo").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/sales.csv")
df = df.na.fill({"revenue": 0})  # replace missing revenue with 0
df.show()

In [0]:
df2 = df.withColumn("revenue_with_tax", df["revenue"] * 1.05)

Spark doesn’t immediately start crunching numbers. Instead, it quietly makes a note in its plan:

“Alright, if I’m ever asked to compute this DataFrame, I’ll need to take the original data, add a new column that multiplies revenue by 1.05, and then return the result.”

This is similar to how an architect works. Imagine you tell an architect you want to build a house with three bedrooms and a balcony. The architect doesn’t run off and start laying bricks. Instead, she adds this detail to the blueprint. Spark does the same — it builds a blueprint of transformations but waits until you give it a reason to execute.

So when does Spark finally stop waiting? When you ask for an action.

### What Counts as an Action?

An action is any instruction that forces Spark to materialize results. It’s like telling the builder, “Okay, now build it.” Until then, Spark is happy to keep sketching.

Some common actions are:

- Asking to see the data (show()).
- Asking to bring the data into Python memory (collect()).
- Asking to count how many rows exist (count()).
- Asking to save the data somewhere (write).

Each one of these actions tells Spark: “Stop planning, and actually run the job.”

Transformations vs Actions

This is where many freshers get tripped up, so let’s pause here.

![](/Volumes/thedataengineering_00/data/sampledata/sampleimages/transformation_action.png)

Transformations are operations that describe how data should change. They don’t execute immediately. Examples are select, filter, withColumn, groupBy. When you call them, Spark builds a new logical plan — a blueprint — but doesn’t touch the data yet.

Actions are operations that demand output. They trigger Spark to actually run the plan. Examples are show, collect, count, take, or writing to disk.

Think of transformations as promises and actions as deliveries. You can make as many promises as you like, but until you ask Spark for delivery, nothing moves.

Let’s See It in Practice

Let’s return to our sales.csv dataset:

In [0]:
from pyspark.sql.functions import col
df2 = df.withColumn("revenue_with_tax", col("revenue") * 1.05)
df2.show()

Everything changes. Spark goes back to the very beginning: it reads the CSV, fills missing revenues with 0, adds the new column, and then finally prints the rows. The action forced Spark to turn the blueprint into execution.

### The Three Actions Every Fresher Should Know
Let’s play with three of the most useful actions:

show()

In [0]:
df.show(5)

In [0]:
rows = df.collect()
print(rows)

This brings the entire DataFrame into Python as a list of Row objects. Powerful, but dangerous — if your dataset is huge, you could run out of memory. Think of collect() as asking Spark to “download everything” into your notebook. Use it only when you know the data is small.

count()

In [0]:
print("Number of rows:", df.count())

### A Small Experiment

Here’s something fun to try:

In [0]:
df2 = df.filter(col("revenue") > 1000)

print("After filter, but before action:")
print(df2)   # just a logical plan, no execution

print("\nNow calling .show():")
df2.show()   # triggers execution

You’ll notice that before the .show(), Spark just tells you the schema. After .show(), it actually runs and gives you the rows. This experiment is the simplest way to feel Spark’s lazy nature.

### Why Laziness is a Good Thing
At first, laziness might seem frustrating. Why can’t Spark just run each step immediately, like pandas? But laziness is actually one of Spark’s superpowers.

By waiting until the last moment, Spark can:

Optimize the plan: it can reorder steps, combine operations, or skip unnecessary work.

Reduce data scans: it only touches data if it’s really needed.

Save time: instead of running five separate steps, it runs one optimized job.

So when Spark delays, it’s not being lazy in the bad sense. It’s being strategic — like a chess player planning three moves ahead.

### Wrapping Up
Today, you learned one of Spark’s most important principles:

Transformations build a plan but don’t execute.

Actions trigger execution and return results.

Spark is lazy by design, so it can optimize your entire pipeline before touching data.

As a fresher, if you ever wonder “Why didn’t Spark run my code?” the answer is probably: “You didn’t call an action.”